[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DavinciDreams/SymbioGPT/blob/main/fuse_pythia.ipynb)

# Cross-Species Symbiogenesis: Pythia-14m → JuliaFluxGPT-fused

**Hypothesis**: A model trained on 300B tokens (Pythia-14m) encodes richer language
structure than one trained on 266M tokens (JuliaSLM). By fusing Pythia's weights
into JuliaFluxGPT-fused alongside the existing JuliaSLM knowledge, we can break
the 3.698 loss plateau.

**Innovation**: Cross-tokenizer fusion ("tokenizer bridge") — mapping BPE-2000 tokens
through Pythia's 50K tokenizer to create compatible embeddings. The tokenizer
is a junction layer, not a wall.

| | Pythia-14m (new source) | JuliaSLM (prev source) | JuliaFluxGPT-fused (target) |
|---|---|---|---|
| d_model | 128 | 256 | 512 |
| Layers | 6 | 6 | 8 |
| Attention | 4H MHA (hd=32) | 4H MHA (hd=64) | 8Q/2KV GQA (hd=64) |
| FFN | GELU 512 | SwiGLU 640 | SwiGLU 1344 |
| Vocab | 50,304 | 2,000 BPE | 2,000 BPE |
| Training | 300B tokens (Pile) | 266M tokens | 115M fine-tune |
| Val loss | — | 3.54 | 3.698 |

GitHub: https://github.com/DavinciDreams/SymbioGPT

In [ ]:
# 1. Setup
!pip install -q wandb huggingface_hub transformers

In [ ]:
# 2. GPU check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'Memory: {mem / 1e9:.1f} GB')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 3. W&B + HF login
import os
import wandb
from huggingface_hub import login as hf_login

try:
    from google.colab import userdata
    os.environ.setdefault('WANDB_API_KEY', userdata.get('WANDB_API_KEY'))
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))
except (ImportError, Exception):
    pass

wandb.login()
hf_login(token=os.environ.get('HF_TOKEN'), add_to_git_credential=False)

In [ ]:
# 4. Download data, BPE-2000 tokenizer, existing fused model, Pythia-14m
import os, sys, math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, HfApi, create_repo
from transformers import AutoModelForCausalLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_REPO = 'LisaMegaWatts/SymbioGPT-10M'
FUSED_REPO = 'LisaMegaWatts/JuliaFluxGPT-fused'
os.makedirs('data', exist_ok=True)

# Download JuliaFluxGPT model code + existing fused checkpoint
print('Downloading JuliaFluxGPT model code and fused checkpoint...')
hf_hub_download(repo_id=FUSED_REPO, filename='juliaflux_model.py', local_dir='.')
hf_hub_download(repo_id=FUSED_REPO, filename='juliaflux_fused_best.pt', local_dir='.')

# Download BPE-2000 tokenizer + curated data
print('Downloading BPE-2000 tokenizer and curated data...')
hf_hub_download(repo_id=DATA_REPO, filename='vocab.json', local_dir='.')
hf_hub_download(repo_id=DATA_REPO, filename='merges.txt', local_dir='.')
hf_hub_download(repo_id=DATA_REPO, filename='data/train_curated.txt.tokens.pt', local_dir='.')
hf_hub_download(repo_id=DATA_REPO, filename='data/val.txt.tokens.pt', local_dir='.')

# Load Pythia-14m-deduped
print('Loading Pythia-14m-deduped (trained on 300B tokens)...')
pythia_model = AutoModelForCausalLM.from_pretrained('EleutherAI/pythia-14m-deduped')
pythia_tokenizer = AutoTokenizer.from_pretrained('EleutherAI/pythia-14m-deduped')
pythia_sd = pythia_model.state_dict()
print(f'Pythia: {sum(p.numel() for p in pythia_model.parameters()):,} params')
print(f'  d={pythia_model.config.hidden_size}, L={pythia_model.config.num_hidden_layers}, '
      f'H={pythia_model.config.num_attention_heads}, '
      f'ffn={pythia_model.config.intermediate_size}')

# Load and chunk tokens
CTX = 256
print('\nLoading tokens...')
train_tokens = torch.load('data/train_curated.txt.tokens.pt', weights_only=True).tolist()
val_tokens = torch.load('data/val.txt.tokens.pt', weights_only=True).tolist()

def chunk(tokens, seq_len):
    n = len(tokens) // (seq_len + 1)
    tokens = tokens[:n * (seq_len + 1)]
    data = torch.tensor(tokens, dtype=torch.long).reshape(n, seq_len + 1)
    return data[:, :-1], data[:, 1:]

train_inputs, train_labels = chunk(train_tokens, CTX)
val_inputs, val_labels = chunk(val_tokens, CTX)
print(f'Train: {len(train_inputs):,} seqs ({len(train_inputs)*CTX:,} tokens)')
print(f'Val: {len(val_inputs):,} seqs')
del train_tokens, val_tokens

In [ ]:
# 5. Tokenizer Bridge: BPE-2000 <-> Pythia-50K
#
# For each BPE-2000 token, decode it to text, re-encode with Pythia's tokenizer,
# and average the corresponding Pythia embeddings. This creates a (2000, 128)
# embedding matrix that bridges the two vocabularies.

def _build_byte_to_unicode():
    """GPT-2 byte-to-unicode mapping."""
    bs = list(range(ord('!'), ord('~') + 1))
    bs += list(range(ord('\xa1'), ord('\xac') + 1))
    bs += list(range(ord('\xae'), ord('\xff') + 1))
    cs = list(bs)
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}


def build_tokenizer_bridge(bpe_vocab, pythia_tok, pythia_embed_in):
    """Map BPE-2000 tokens -> Pythia embedding space via text alignment.
    
    Args:
        bpe_vocab: dict {token_str: token_id} from vocab.json
        pythia_tok: Pythia's HF tokenizer
        pythia_embed_in: Pythia's input embedding weight (50304, 128)
    Returns:
        (2000, 128) tensor of bridged embeddings
    """
    byte_to_unicode = _build_byte_to_unicode()
    unicode_to_byte = {v: k for k, v in byte_to_unicode.items()}
    
    vocab_size = len(bpe_vocab)
    d_pythia = pythia_embed_in.shape[1]
    bridged = torch.zeros(vocab_size, d_pythia)
    
    mapped = 0
    fallback = 0
    
    for token_str, token_id in bpe_vocab.items():
        # Special tokens
        if token_str.startswith('<|') and token_str.endswith('|>'):
            bridged[token_id] = torch.randn(d_pythia) * 0.02
            fallback += 1
            continue
        
        # Decode BPE-2000 unicode token -> raw bytes -> text
        raw_bytes = bytearray()
        for ch in token_str:
            b = unicode_to_byte.get(ch)
            if b is not None:
                raw_bytes.append(b)
            else:
                raw_bytes.extend(ch.encode('utf-8'))
        text = raw_bytes.decode('utf-8', errors='replace')
        
        # Encode text with Pythia's tokenizer
        pythia_ids = pythia_tok.encode(text, add_special_tokens=False)
        
        if len(pythia_ids) > 0:
            # Average Pythia embeddings for corresponding tokens
            valid_ids = [i for i in pythia_ids if i < pythia_embed_in.shape[0]]
            if valid_ids:
                embeds = pythia_embed_in[valid_ids]  # (N, 128)
                bridged[token_id] = embeds.mean(dim=0)
                mapped += 1
            else:
                bridged[token_id] = torch.randn(d_pythia) * 0.02
                fallback += 1
        else:
            bridged[token_id] = torch.randn(d_pythia) * 0.02
            fallback += 1
    
    print(f'Tokenizer bridge: {mapped}/{vocab_size} mapped, {fallback} fallback')
    return bridged


# Load BPE-2000 vocab
with open('vocab.json') as f:
    bpe_vocab = json.load(f)
print(f'BPE-2000 vocab: {len(bpe_vocab)} tokens')

# Build bridge
pythia_embed_in = pythia_sd['gpt_neox.embed_in.weight'].float()
bridged_embeddings = build_tokenizer_bridge(bpe_vocab, pythia_tokenizer, pythia_embed_in)

# Sanity check: show a few mappings
byte_to_unicode = _build_byte_to_unicode()
unicode_to_byte = {v: k for k, v in byte_to_unicode.items()}
print('\nSample token mappings:')
for tok_str in ['the', 'Ġthe', 'Ġof', 'ĠPhil', 'osophy', '.', ',']:
    if tok_str in bpe_vocab:
        raw = bytearray(unicode_to_byte.get(c, ord(c)) for c in tok_str)
        text = raw.decode('utf-8', errors='replace')
        pythia_ids = pythia_tokenizer.encode(text, add_special_tokens=False)
        pythia_toks = [pythia_tokenizer.decode([i]) for i in pythia_ids]
        print(f'  BPE "{tok_str}" (id={bpe_vocab[tok_str]}) -> text "{text}" '
              f'-> Pythia {pythia_ids} {pythia_toks}')

print(f'\nBridged embeddings: {bridged_embeddings.shape}')
print(f'Mean norm: {bridged_embeddings.norm(dim=1).mean():.4f}')

In [ ]:
# 6. Weight projection: Pythia-14m (d=128, 6L, 4H MHA) -> JuliaFluxGPT (d=512, 8L, 8Q/2KV GQA)
#
# Adaptations:
# - Vocab: 50K -> 2K via tokenizer bridge (cell 5)
# - d_model: 128 -> 512 (zero-pad + noise)
# - Attention: fused QKV [384,128] -> separate wq [512,512] + wkv [256,512]
#   - head_dim: 32 -> 64 (zero-pad)
#   - 4 MHA heads -> 8Q/2KV GQA (duplicate + average)
# - FFN: GELU 2-matrix -> SwiGLU 3-matrix (duplicate up -> gate+up)
# - Norm: LayerNorm -> RMSNorm (copy weight, drop bias)
# - Bias: drop all biases
# - Layers: 6 -> 8 (random init layers 6-7)

sys.path.insert(0, '.')
from juliaflux_model import JuliaFluxConfig, JuliaFluxGPT

config = JuliaFluxConfig(
    d_model=512, n_layers=8, n_heads=8, n_kv_heads=2,
    head_dim=64, context_length=CTX, vocab_size=2000,
    weight_tying=True, rope_base=10000.0,
)

# Pythia dimensions
P_D = 128       # d_model
P_H = 4         # n_heads
P_HD = 32       # head_dim
P_FFN = 512     # intermediate_size
P_L = 6         # n_layers

# Target dimensions
T_D = config.d_model        # 512
T_H = config.n_heads        # 8 (query heads)
T_KV = config.n_kv_heads    # 2
T_HD = config.head_dim      # 64
T_KV_DIM = T_KV * T_HD     # 128
raw_inner = int(4 * T_D * 2 / 3)
T_FFN = max(64, 64 * ((raw_inner + 32) // 64))  # 1344


def project_pythia_to_juliafluxgpt(pythia_sd, bridged_emb, config):
    """Project Pythia-14m weights into a fresh JuliaFluxGPT model.
    
    Returns the model with transferred weights.
    """
    model = JuliaFluxGPT(config)
    sd = model.state_dict()
    
    # === Embedding: bridged (2000, 128) -> (2000, 512) ===
    emb_std = bridged_emb.std().item()
    sd['tok_emb.weight'][:, :P_D] = bridged_emb
    sd['tok_emb.weight'][:, P_D:] = torch.randn(2000, T_D - P_D) * emb_std * 0.02
    print(f'  Embedding: bridged ({bridged_emb.shape}) -> ({sd["tok_emb.weight"].shape})')
    
    # === Per-layer projection ===
    for i in range(P_L):
        pfx_p = f'gpt_neox.layers.{i}'  # Pythia key prefix
        pfx_t = f'blocks.{i}'            # Target key prefix
        
        # --- RMSNorm (from LayerNorm: copy weight, drop bias, pad with 1.0) ---
        # Pythia has input_layernorm and post_attention_layernorm
        # Target has ln1 (pre-attn) and ln2 (pre-ffn)
        src_ln1 = pythia_sd[f'{pfx_p}.input_layernorm.weight'].float()
        sd[f'{pfx_t}.ln1.weight'][:P_D] = src_ln1
        sd[f'{pfx_t}.ln1.weight'][P_D:] = 1.0
        
        src_ln2 = pythia_sd[f'{pfx_p}.post_attention_layernorm.weight'].float()
        sd[f'{pfx_t}.ln2.weight'][:P_D] = src_ln2
        sd[f'{pfx_t}.ln2.weight'][P_D:] = 1.0
        
        # --- Attention: fused QKV (384, 128) -> wq (512, 512) + wkv (256, 512) ---
        # Pythia QKV layout: [Q_all(128) | K_all(128) | V_all(128)] = (384, 128)
        qkv_w = pythia_sd[f'{pfx_p}.attention.query_key_value.weight'].float()  # (384, 128)
        src_q = qkv_w[:P_H * P_HD, :]     # (128, 128) = 4 heads * 32
        src_k = qkv_w[P_H * P_HD:2 * P_H * P_HD, :]  # (128, 128)
        src_v = qkv_w[2 * P_H * P_HD:, :]              # (128, 128)
        
        # wq: each of 4 Pythia heads -> 2 target Q heads
        # head_dim: 32 -> 64 (zero-pad upper half)
        wq_key = f'{pfx_t}.attn.wq.weight'
        sd[wq_key].zero_()
        for j in range(P_H):  # 4 source heads
            head_rows = src_q[j * P_HD:(j + 1) * P_HD, :]  # (32, 128)
            # Target head 2j: first P_HD rows of T_HD-sized head, first P_D input cols
            t_start_a = (2 * j) * T_HD
            t_start_b = (2 * j + 1) * T_HD
            sd[wq_key][t_start_a:t_start_a + P_HD, :P_D] = head_rows
            sd[wq_key][t_start_b:t_start_b + P_HD, :P_D] = head_rows  # duplicate
        
        # wkv: pairs of Pythia K/V heads averaged -> target KV heads
        # Layout: [K_h0(64) | K_h1(64) | V_h0(64) | V_h1(64)] = (256, 512)
        wkv_key = f'{pfx_t}.attn.wkv.weight'
        sd[wkv_key].zero_()
        for h in range(T_KV):  # 2 target KV heads
            # Average pairs of source heads
            k_avg = (src_k[2 * h * P_HD:(2 * h + 1) * P_HD, :] +
                     src_k[(2 * h + 1) * P_HD:(2 * h + 2) * P_HD, :]) / 2  # (32, 128)
            v_avg = (src_v[2 * h * P_HD:(2 * h + 1) * P_HD, :] +
                     src_v[(2 * h + 1) * P_HD:(2 * h + 2) * P_HD, :]) / 2  # (32, 128)
            # K section
            sd[wkv_key][h * T_HD:h * T_HD + P_HD, :P_D] = k_avg
            # V section (offset by T_KV_DIM)
            sd[wkv_key][T_KV_DIM + h * T_HD:T_KV_DIM + h * T_HD + P_HD, :P_D] = v_avg
        
        # proj (wo): (128, 128) -> (512, 512) with head-aware split
        src_wo = pythia_sd[f'{pfx_p}.attention.dense.weight'].float()  # (128, 128)
        proj_key = f'{pfx_t}.attn.proj.weight'
        sd[proj_key].zero_()
        for j in range(P_H):  # 4 source heads
            col_slice = src_wo[:, j * P_HD:(j + 1) * P_HD]  # (128, 32)
            # Duplicate to target head pairs, split by 2 for magnitude preservation
            t_col_a = (2 * j) * T_HD
            t_col_b = (2 * j + 1) * T_HD
            sd[proj_key][:P_D, t_col_a:t_col_a + P_HD] = col_slice / 2
            sd[proj_key][:P_D, t_col_b:t_col_b + P_HD] = col_slice / 2
        
        # --- FFN: GELU 2-matrix -> SwiGLU 3-matrix ---
        # Pythia: dense_h_to_4h (512, 128) + dense_4h_to_h (128, 512)
        # Target: w_gate (1344, 512) + w_up (1344, 512) + w_down (512, 1344)
        src_up = pythia_sd[f'{pfx_p}.mlp.dense_h_to_4h.weight'].float()  # (512, 128)
        src_down = pythia_sd[f'{pfx_p}.mlp.dense_4h_to_h.weight'].float()  # (128, 512)
        
        # w_gate and w_up both get the up-projection (they'll differentiate during fine-tuning)
        sd[f'{pfx_t}.ffn.w_gate.weight'].zero_()
        sd[f'{pfx_t}.ffn.w_gate.weight'][:P_FFN, :P_D] = src_up
        sd[f'{pfx_t}.ffn.w_up.weight'].zero_()
        sd[f'{pfx_t}.ffn.w_up.weight'][:P_FFN, :P_D] = src_up
        
        # w_down
        sd[f'{pfx_t}.ffn.w_down.weight'].zero_()
        sd[f'{pfx_t}.ffn.w_down.weight'][:P_D, :P_FFN] = src_down
        
        print(f'  Layer {i}: QKV split, Q dup x2, KV avg, proj split/2, '
              f'FFN {P_FFN}->{T_FFN}, LN->RMS')
    
    # Layers 6-7: random init
    for i in range(P_L, config.n_layers):
        print(f'  Layer {i}: random init (no Pythia source)')
    
    # Final norm
    src_ln_f = pythia_sd['gpt_neox.final_layer_norm.weight'].float()
    sd['ln_f.weight'][:P_D] = src_ln_f
    sd['ln_f.weight'][P_D:] = 1.0
    
    model.load_state_dict(sd)
    total = sum(p.numel() for p in model.parameters())
    print(f'\nProjection complete: Pythia-14m -> JuliaFluxGPT ({total:,} params)')
    return model


print('Projecting Pythia-14m -> JuliaFluxGPT...')
print(f'  Source: d={P_D}, {P_L}L, {P_H}H MHA (hd={P_HD}), ffn={P_FFN}, vocab=50304')
print(f'  Target: d={T_D}, {config.n_layers}L, {T_H}Q/{T_KV}KV GQA (hd={T_HD}), ffn={T_FFN}, vocab=2000')
print()
pythia_projected = project_pythia_to_juliafluxgpt(pythia_sd, bridged_embeddings, config)

# Free Pythia from memory
del pythia_model, pythia_sd, pythia_embed_in
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# 7. Blend Pythia projection with existing JuliaSLM-fused weights
#
# JuliaFluxGPT-fused already has JuliaSLM knowledge (val_loss=3.698).
# We blend Pythia projections with existing weights:
#   - Layers 0-5: alpha blend (preserve JuliaSLM + add Pythia)
#   - Layers 6-7: Pythia only (these were random init in JuliaSLM fusion)
#   - Embeddings + final norm: alpha blend

ALPHA = 0.5  # blend ratio: higher = more existing, lower = more Pythia

print(f'Loading existing JuliaFluxGPT-fused checkpoint...')
existing_ckpt = torch.load('juliaflux_fused_best.pt', map_location='cpu', weights_only=True)
# Handle both formats: raw state_dict or wrapped dict
if isinstance(existing_ckpt, dict) and 'model_state_dict' in existing_ckpt:
    existing_sd = existing_ckpt['model_state_dict']
else:
    existing_sd = existing_ckpt
print(f'  Loaded {len(existing_sd)} weight tensors')

# Blend
pythia_proj_sd = pythia_projected.state_dict()
blended_sd = {}

for key in pythia_proj_sd:
    if key not in existing_sd:
        # Key only in Pythia projection
        blended_sd[key] = pythia_proj_sd[key]
        continue
    
    # Determine blend alpha based on layer
    layer_alpha = ALPHA
    # Layers 6-7 were randomly initialized in JuliaSLM fusion -> use Pythia only
    if any(f'blocks.{i}.' in key for i in [6, 7]):
        layer_alpha = 0.0  # all Pythia
    
    blended_sd[key] = layer_alpha * existing_sd[key].float() + (1 - layer_alpha) * pythia_proj_sd[key].float()

# Load blended weights into a fresh model
model = JuliaFluxGPT(config)
model.load_state_dict(blended_sd)
model = model.to(device)

# Verify: no NaN/Inf
model.eval()
with torch.no_grad():
    test_out = model(val_inputs[:4].to(device))
    assert not torch.isnan(test_out).any(), 'NaN in blended model!'
    assert not torch.isinf(test_out).any(), 'Inf in blended model!'
print('Forward pass: OK (no NaN/Inf)')

print(f'\nBlend strategy (alpha={ALPHA}):')
print(f'  Layers 0-5: {ALPHA:.0%} existing + {1-ALPHA:.0%} Pythia')
print(f'  Layers 6-7: 100% Pythia (were random init)')
print(f'  Embeddings: {ALPHA:.0%} existing + {1-ALPHA:.0%} bridged Pythia')

del pythia_projected, existing_sd, existing_ckpt, blended_sd
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# 8. Evaluate baselines

@torch.no_grad()
def evaluate(model, val_inputs, val_labels, batch_size=128):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    dev = next(model.parameters()).device
    for i in range(0, len(val_inputs), batch_size):
        batch_in = val_inputs[i:i+batch_size].to(dev)
        batch_tgt = val_labels[i:i+batch_size].to(dev)
        logits = model(batch_in)
        B, T, V = logits.shape
        loss = F.cross_entropy(
            logits.float().reshape(B*T, V), batch_tgt.reshape(B*T), reduction='sum'
        )
        total_loss += loss.item()
        total_tokens += B * T
    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl

blend_loss, blend_ppl = evaluate(model, val_inputs, val_labels)
print(f'Blended model (before fine-tune): val_loss={blend_loss:.4f} ppl={blend_ppl:.1f}')
print(f'  (random init would be ~7.6, existing fused was 3.698)')

print(f'\n{"Model":<40} {"Val Loss":>10} {"PPL":>8}')
print('-' * 60)
print(f'{"JuliaSLM (source, d=256)":<40} {"3.5400":>10} {"34.5":>8}')
print(f'{"JuliaFluxGPT-fused v1 (JuliaSLM only)":<40} {"3.6980":>10} {"40.4":>8}')
print(f'{"Blended (before fine-tune)":<40} {blend_loss:>10.4f} {blend_ppl:>8.1f}')

In [ ]:
# 9. Fine-tune

BATCH_SIZE = 64
TOTAL_STEPS = 10_000
WARMUP = 300
LR = 3e-4
MIN_LR = 1e-5
EVAL_INTERVAL = 500

HF_REPO = 'LisaMegaWatts/JuliaFluxGPT-fused-v2'

n_params = sum(p.numel() for p in model.parameters())
print(f'Fine-tuning: {n_params:,} params, {TOTAL_STEPS} steps')
print(f'  LR: {LR} -> {MIN_LR} (cosine, {WARMUP} warmup)')
print(f'  Batch: {BATCH_SIZE} x {CTX} = {BATCH_SIZE * CTX:,} tok/step')

run = wandb.init(
    project='symbiogenesis',
    name='juliafluxgpt-pythia-fusion',
    config={
        'model': 'JuliaFluxGPT-fused-v2',
        'method': 'cross_species_symbiogenesis',
        'source_new': 'Pythia-14m-deduped (d=128, 300B tokens)',
        'source_existing': 'JuliaSLM (d=256, val_loss=3.54)',
        'blend_alpha': ALPHA,
        'total_params': n_params,
        'batch_size': BATCH_SIZE,
        'total_steps': TOTAL_STEPS,
        'lr': LR,
        'pre_finetune_loss': blend_loss,
    },
    tags=['fusion', 'cross-species', 'pythia', 'tokenizer-bridge', 'symbiogenesis'],
    reinit='finish_previous',
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))

def lr_schedule(step):
    if step < WARMUP:
        return (step + 1) / max(WARMUP, 1)
    progress = (step - WARMUP) / max(TOTAL_STEPS - WARMUP, 1)
    return max(MIN_LR / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

n_train = len(train_inputs)
model.train()
best_loss = float('inf')
best_step = 0
history = []
t_start = time.time()
step = 0

while step < TOTAL_STEPS:
    perm = torch.randperm(n_train)
    for i in range(0, n_train, BATCH_SIZE):
        if step >= TOTAL_STEPS:
            break
        idx = perm[i:i + BATCH_SIZE]
        batch_in = train_inputs[idx].to(device)
        batch_tgt = train_labels[idx].to(device)

        logits = model(batch_in)
        B, T, V = logits.shape
        loss = F.cross_entropy(logits.reshape(B * T, V), batch_tgt.reshape(B * T))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()

        if step % 50 == 0:
            wandb.log({'train/loss': loss.item(),
                       'train/lr': scheduler.get_last_lr()[0]}, step=step)

        if step > 0 and step % EVAL_INTERVAL == 0:
            val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
            history.append((step, val_loss, val_ppl))
            wandb.log({'val/loss': val_loss, 'val/ppl': val_ppl}, step=step)
            marker = ' ** BEST **' if val_loss < best_loss else ''
            if val_loss < best_loss:
                best_loss = val_loss
                best_step = step
                torch.save(model.state_dict(), 'juliaflux_fused_v2_best.pt')
            elapsed = time.time() - t_start
            print(f'  [step {step:5d}] val={val_loss:.4f} ppl={val_ppl:.1f} ({elapsed:.0f}s){marker}')
            model.train()

        step += 1

# Final eval
val_loss, val_ppl = evaluate(model, val_inputs, val_labels)
history.append((step, val_loss, val_ppl))
if val_loss < best_loss:
    best_loss = val_loss
    best_step = step
    torch.save(model.state_dict(), 'juliaflux_fused_v2_best.pt')

elapsed = time.time() - t_start
print(f'\nDone ({elapsed:.0f}s): best val_loss={best_loss:.4f} at step {best_step}')
wandb.log({'val/final_loss': best_loss})
wandb.finish()

In [ ]:
# 10. Results + HF upload

print('\n' + '=' * 70)
print('CROSS-SPECIES SYMBIOGENESIS RESULTS')
print('=' * 70)

print(f'\n{"Model":<45} {"Params":>10} {"Val Loss":>10} {"PPL":>8}')
print('-' * 75)
print(f'{"JuliaSLM (prev source)":<45} {"5.04M":>10} {"3.5400":>10} {"34.5":>8}')
print(f'{"JuliaFluxGPT-fused v1 (JuliaSLM only)":<45} {"22.79M":>10} {"3.6980":>10} {"40.4":>8}')
print(f'{"Blended (before fine-tune)":<45} {"22.79M":>10} {blend_loss:>10.4f} {blend_ppl:>8.1f}')
print(f'{"** JuliaFluxGPT-fused v2 (+ Pythia) **":<45} {"22.79M":>10} {best_loss:>10.4f} {math.exp(min(best_loss,20)):>8.1f}')

print(f'\nScaling context (curated data, BPE-2000, ctx=256):')
print(f'  SymbioSLM      4.07M -> 3.62')
print(f'  JuliaSLM       5.04M -> 3.54')
print(f'  SymbioGPT-10M 11.05M -> 3.56')
print(f'  FluxGPT-fused 22.79M -> {best_loss:.4f}  (Pythia + JuliaSLM fusion)')

# Upload to HF
try:
    hf_api = HfApi()
    create_repo(HF_REPO, exist_ok=True)
    if os.path.exists('juliaflux_fused_v2_best.pt'):
        hf_api.upload_file(
            path_or_fileobj='juliaflux_fused_v2_best.pt',
            path_in_repo='juliaflux_fused_v2_best.pt',
            repo_id=HF_REPO,
            commit_message=f'Cross-species fusion: val_loss={best_loss:.4f}',
        )
    print(f'\nUploaded to: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF upload failed: {e}')

print('\nDone!')